# Loan Dataset: Ensemble Classification and Regression Modelling

This notebook contains two parts:

- Part A: Ensemble classification for Loan Approval Status
- Part B: Decision Tree regression for Maximum Loan Amount

All outputs are formatted clearly for coursework screenshots.

## 1. Import Required Libraries
Import all packages needed for preprocessing, modelling, evaluation, and plotting.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style("whitegrid")

## 2. Load Or Reuse Preprocessed Datasets
This notebook reuses existing X_class, y_class, X_reg, and y_reg if available in memory.
If not available, it rebuilds them from the CSV file using the same preprocessing logic.

In [ ]:
datasets_available = ("X_class" in globals()) and ("y_class" in globals()) and ("X_reg" in globals()) and ("y_reg" in globals())

if not datasets_available:
    print("Preprocessed datasets not found in memory. Rebuilding from CSV...")
    csv_path = "loan_approval_data.csv"
    df = pd.read_csv(csv_path)

    def normalize_name(text):
        return str(text).strip().lower().replace(" ", "").replace("_", "")

    id_like_columns = []
    for col in df.columns:
        normalized_col = normalize_name(col)
        if normalized_col == "id" or normalized_col.endswith("id") or normalized_col in {"loanid", "applicationid"}:
            id_like_columns.append(col)
    df = df.drop(columns=id_like_columns, errors="ignore")

    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = df.select_dtypes(exclude=[np.number]).columns.tolist()
    for col in numeric_columns:
        df[col] = df[col].fillna(df[col].mean())
    for col in categorical_columns:
        df[col] = df[col].fillna(df[col].mode(dropna=True)[0])

    normalized_to_original = {normalize_name(c): c for c in df.columns}
    class_target_candidates = ["loan_approval_status", "Loan Approval Status", "loan_status", "approved"]
    reg_target_candidates = ["max_allowed_loan", "Maximum Loan Amount", "maximum_loan_amount", "loan_amount_max"]

    class_target_col = None
    for candidate in class_target_candidates:
        candidate_key = normalize_name(candidate)
        if candidate_key in normalized_to_original:
            class_target_col = normalized_to_original[candidate_key]
            break

    reg_target_col = None
    for candidate in reg_target_candidates:
        candidate_key = normalize_name(candidate)
        if candidate_key in normalized_to_original:
            reg_target_col = normalized_to_original[candidate_key]
            break

    if class_target_col is None:
        raise ValueError("Could not find Loan Approval Status target column.")
    if reg_target_col is None:
        raise ValueError("Could not find Maximum Loan Amount target column.")

    selected_feature_columns = [
        col for col in df.columns
        if col not in {class_target_col, reg_target_col}
    ]

    X_class = df[selected_feature_columns].copy()
    y_class = df[class_target_col].copy()
    X_class = pd.get_dummies(X_class, drop_first=False)
    if not pd.api.types.is_numeric_dtype(y_class):
        label_encoder = LabelEncoder()
        y_class = pd.Series(label_encoder.fit_transform(y_class), name=class_target_col)

    approval_series = df[class_target_col].copy()
    approved_value = 1
    if not pd.api.types.is_numeric_dtype(approval_series):
        unique_values = [str(v).strip().lower() for v in approval_series.dropna().unique().tolist()]
        approval_candidates = ["approved", "approve", "yes", "y", "true", "1"]
        matched_values = [v for v in unique_values if v in approval_candidates]
        if len(matched_values) > 0:
            approved_value = matched_values[0]
        approval_series = approval_series.astype(str).str.strip().str.lower()

    if pd.api.types.is_numeric_dtype(df[class_target_col]):
        class_means = df.groupby(class_target_col)[reg_target_col].mean()
        approved_value = class_means.idxmax()
        approval_series = df[class_target_col]

    approved_df = df[approval_series == approved_value].copy()
    if approved_df.shape[0] == 0:
        raise ValueError("No approved loans found for regression dataset.")

    X_reg = approved_df.drop(columns=[reg_target_col, class_target_col], errors="ignore").copy()
    y_reg = approved_df[reg_target_col].copy()
    X_reg = pd.get_dummies(X_reg, drop_first=False)

if datasets_available:
    print("Using existing preprocessed X_class, y_class, X_reg, and y_reg from memory.")

X_class = pd.DataFrame(X_class).reset_index(drop=True)
y_class = pd.Series(y_class).reset_index(drop=True)
X_reg = pd.DataFrame(X_reg).reset_index(drop=True)
y_reg = pd.Series(y_reg).reset_index(drop=True)

print("X_class shape:", X_class.shape)
print("y_class shape:", y_class.shape)
print("X_reg shape:", X_reg.shape)
print("y_reg shape:", y_reg.shape)

## Part A: Ensemble Classifier

This section builds two base learners, combines them with soft voting, evaluates all models, and compares ensemble performance to individual models.

### A1. Train-Test Split For Classification
Use an 80/20 split with stratification to preserve class balance.

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    stratify=y_class,
    random_state=42
)

print("Xc_train shape:", Xc_train.shape)
print("Xc_test shape:", Xc_test.shape)
print("yc_train shape:", yc_train.shape)
print("yc_test shape:", yc_test.shape)

### A2. Build Base Learners And Soft Voting Ensemble
Base learners used: Logistic Regression and KNN.
Then combine them using VotingClassifier with soft voting.

In [ ]:
log_reg_base = LogisticRegression(max_iter=2000, random_state=42)
knn_base = KNeighborsClassifier(n_neighbors=5)

voting_ensemble = VotingClassifier(
    estimators=[("lr", LogisticRegression(max_iter=2000, random_state=42)), ("knn", KNeighborsClassifier(n_neighbors=5))],
    voting="soft"
)

classification_models = {
    "Logistic Regression": log_reg_base,
    "KNN (k=5)": knn_base,
    "Voting Ensemble (Soft)": voting_ensemble
}

for model_name, model_obj in classification_models.items():
    model_obj.fit(Xc_train, yc_train)
    print(f"Trained model: {model_name}")

### A3. Evaluate Individual Models And Ensemble
For each model, display confusion matrix, classification report, ROC curve, and AUC.

In [ ]:
class_labels = np.sort(np.unique(yc_test))
if len(class_labels) != 2:
    raise ValueError("Binary target is required for this ROC/AUC setup.")

positive_class = class_labels[-1]
ensemble_comparison_rows = []

roc_fig, roc_ax = plt.subplots(figsize=(8, 6))

for model_name, model_obj in classification_models.items():
    print("\n" + "=" * 90)
    print(f"Evaluation: {model_name}")
    print("=" * 90)

    y_pred = model_obj.predict(Xc_test)
    y_prob = model_obj.predict_proba(Xc_test)[:, 1]

    cm = confusion_matrix(yc_test, y_pred)
    print("Confusion Matrix:")
    display(pd.DataFrame(cm, index=["Actual 0", "Actual 1"], columns=["Pred 0", "Pred 1"]))

    cm_fig, cm_ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=cm_ax)
    cm_ax.set_title(f"Confusion Matrix - {model_name}")
    cm_ax.set_xlabel("Predicted Label")
    cm_ax.set_ylabel("True Label")
    cm_fig.tight_layout()
    display(cm_fig)
    plt.close(cm_fig)

    report_dict = classification_report(yc_test, y_pred, output_dict=True, zero_division=0)
    print("Classification Report:")
    display(pd.DataFrame(report_dict).transpose().round(4))

    fpr, tpr, _ = roc_curve(yc_test, y_prob, pos_label=positive_class)
    auc_value = roc_auc_score(yc_test, y_prob)
    roc_ax.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {auc_value:.4f})")

    positive_recall = report_dict[str(int(positive_class))]["recall"] if str(int(positive_class)) in report_dict else report_dict[str(positive_class)]["recall"]
    ensemble_comparison_rows.append({
        "Model": model_name,
        "Recall": positive_recall,
        "AUC": auc_value
    })

roc_ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
roc_ax.set_title("ROC Curve Comparison: Individual Models vs Ensemble")
roc_ax.set_xlabel("False Positive Rate")
roc_ax.set_ylabel("True Positive Rate")
roc_ax.legend(loc="lower right")
roc_fig.tight_layout()
display(roc_fig)
plt.close(roc_fig)

### A4. Compare Ensemble Performance With Individual Models
Sort and compare models by Recall and AUC to check whether the ensemble improves classification quality.

In [ ]:
ensemble_results_df = pd.DataFrame(ensemble_comparison_rows)
ensemble_results_df = ensemble_results_df.sort_values(by=["Recall", "AUC"], ascending=False).reset_index(drop=True)

print("Classification Model Comparison (Recall and AUC):")
display(ensemble_results_df.round(4))

## Part B: Regression Modelling

This section trains and compares two Decision Tree Regressors on Maximum Loan Amount prediction.

### B1. Train-Test Split For Regression
Use an 80/20 split for regression dataset.

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Xr_train shape:", Xr_train.shape)
print("Xr_test shape:", Xr_test.shape)
print("yr_train shape:", yr_train.shape)
print("yr_test shape:", yr_test.shape)

### B2. Build And Train Two Decision Tree Regressors
- DT1: Default fully grown tree
- DT2: Pruned tree with max_depth = 4

In [ ]:
dt1_model = DecisionTreeRegressor(random_state=42)
dt2_model = DecisionTreeRegressor(max_depth=4, random_state=42)

dt1_model.fit(Xr_train, yr_train)
dt2_model.fit(Xr_train, yr_train)

print("Trained DT1 and DT2 models successfully.")

### B3. Evaluate DT1 And DT2
Evaluate both models using MSE, MAE, and R2 score, then compare results.

In [ ]:
yr_pred_dt1 = dt1_model.predict(Xr_test)
yr_pred_dt2 = dt2_model.predict(Xr_test)

mse_dt1 = mean_squared_error(yr_test, yr_pred_dt1)
mae_dt1 = mean_absolute_error(yr_test, yr_pred_dt1)
r2_dt1 = r2_score(yr_test, yr_pred_dt1)

mse_dt2 = mean_squared_error(yr_test, yr_pred_dt2)
mae_dt2 = mean_absolute_error(yr_test, yr_pred_dt2)
r2_dt2 = r2_score(yr_test, yr_pred_dt2)

regression_comparison_df = pd.DataFrame({
    "Model": ["DT1 (Default)", "DT2 (max_depth=4)"],
    "MSE": [mse_dt1, mse_dt2],
    "MAE": [mae_dt1, mae_dt2],
    "R2": [r2_dt1, r2_dt2]
})

print("Decision Tree Regression Comparison:")
display(regression_comparison_df.round(4))

### B4. Visualize DT1 And DT2 Trees
Plot both decision trees for interpretability.

For DT1, only the top levels are shown to keep the plot readable.

In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(
    dt1_model,
    feature_names=Xr_train.columns,
    filled=True,
    rounded=True,
    max_depth=3,
    fontsize=8
)
plt.title("DT1 (Default Fully Grown) - Top 3 Levels")
plt.tight_layout()
plt.show()

plt.figure(figsize=(24, 10))
plot_tree(
    dt2_model,
    feature_names=Xr_train.columns,
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("DT2 (Pruned) - max_depth = 4")
plt.tight_layout()
plt.show()

### B5. Predict Maximum Loan Amount For A New Sample Client
Create a sample client row based on an existing feature row and predict with both DT1 and DT2.

In [ ]:
new_sample_client = Xr_train.iloc[[0]].copy()

if "age" in new_sample_client.columns:
    new_sample_client["age"] = 32
if "income" in new_sample_client.columns:
    new_sample_client["income"] = 78000
if "loan_amount" in new_sample_client.columns:
    new_sample_client["loan_amount"] = 18000

print("New Sample Client Features:")
display(new_sample_client)

predicted_loan_dt1 = dt1_model.predict(new_sample_client)[0]
predicted_loan_dt2 = dt2_model.predict(new_sample_client)[0]
predicted_loan_avg = (predicted_loan_dt1 + predicted_loan_dt2) / 2

print(f"Predicted Maximum Loan Amount by DT1: {predicted_loan_dt1:.2f}")
print(f"Predicted Maximum Loan Amount by DT2: {predicted_loan_dt2:.2f}")
print(f"Average of DT1 and DT2 Predictions: {predicted_loan_avg:.2f}")

## Final Summary
This notebook completed:

- Ensemble classification with soft voting and full evaluation
- Comparison of ensemble against individual base learners
- Regression modelling with two decision trees
- Metric comparison for DT1 and DT2
- Tree visualizations
- Prediction for a new sample client

All outputs are displayed clearly for coursework screenshots.